In [ ]:
import numpy as np
import torch
import os
import pickle
import gzip
from components.other_utilities.models_to_train import ResNetPLModel
from components.FL_sim import FLSimulator
from components.other_utilities.datasets import FasterSVHN
from torchvision import transforms

# %%
w0_accu_grads_list = None
w1_accu_grads_list = None
res_per_step = None
batch_count = None
client_epochs_per_round = None

min_window_size = 10
def compute_corr(worker_id, round_number, accum_grads,):
    max_window_size = int(batch_count*client_epochs_per_round*1.2)
    flatten_vec = torch.concatenate([v.flatten().cpu() for v in accum_grads.values()]).to(torch.float16)
    if worker_id == 0:
        w0_accu_grads_list.append(flatten_vec)
        return
    elif worker_id == 1:
        w1_accu_grads_list.append(flatten_vec)

    if len(w1_accu_grads_list) < min_window_size:
        return

    if len(w1_accu_grads_list) > max_window_size:
        # replace the oldest vector with none to save memory
        w0_accu_grads_list[len(w1_accu_grads_list)-(max_window_size+1)] = None
        w1_accu_grads_list[-(max_window_size+1)] = None

    temp = len(w1_accu_grads_list) - min(len(w1_accu_grads_list), max_window_size)
    vectors_to_check = torch.stack([
        torch.stack(w0_accu_grads_list[temp:len(w1_accu_grads_list)]).T,
        torch.stack(w1_accu_grads_list[temp:]).T,
    ])
    vectors_to_check = torch.transpose(vectors_to_check, 0, 1)

    # compute the correlation for each weight between workers
    corr_list = torch.func.vmap(torch.corrcoef)(vectors_to_check)[:, 0, 1].to(torch.float32)
    temp = torch.isnan(corr_list)
    summed_non_nan_corr = corr_list[~temp].abs().mean().cpu().numpy()

    res_per_step.append((summed_non_nan_corr, round_number, temp.sum()/len(corr_list)))

    # write the last computed correlation to disk
    # iterate between 2 files to avoid corruption
    file_name = f'corr_res_{len(res_per_step)%2}.pkl.gz'
    with gzip.open(file_name, 'wb', compresslevel=1) as f:
        pickle.dump(res_per_step, f)


class ModeUsedForFL(ResNetPLModel):
    def on_before_optimizer_step(self, optimizer):
        super().on_before_optimizer_step(optimizer)

        worker_id = self.current_step_info['worker_id']
        curr_round = self.current_step_info['curr_round']
        current_epoch = self.current_step_info['current_epoch']
        batch_idx = self.current_step_info['batch_idx']

        # write the accum grad to disk
        accum_grad = {k: v.detach().clone() for k, v in self.accu_param_grads.items()}
        round_number = curr_round + (current_epoch + batch_idx / batch_count)/client_epochs_per_round
        compute_corr(worker_id, round_number, accum_grad)

    def clone(self, copy=None):
        if copy is None:
            copy = self.__class__(num_classes=self.num_classes, lr=self.lr, resnet_version=self.resnet_version)
        return super(ResNetPLModel, self).clone(copy=copy)


# %%
__batch_size__ = 7_500  # 3 batches per w = 73000/2worker/13000  -  np.ceil(len(dataset[0])/batch_size/2)
__epoch_count__ = 10
if __name__ == '__main__':
    torch.set_float32_matmul_precision('medium')

    import logging
    import warnings
    logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)
    logger = logging.getLogger('pytorch_lightning.utilities.rank_zero')
    logger.setLevel(logging.ERROR)
    warnings.filterwarnings("ignore", "LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]")
    warnings.filterwarnings("ignore", "The 'train_dataloader' does")
    warnings.filterwarnings("ignore", "You defined a `validation_step` but")
    warnings.filterwarnings("ignore", "Starting from v1.9.0, `tensorboardX` has been")
    warnings.filterwarnings("ignore", "The number of training batches")
    warnings.filterwarnings("ignore", "`Trainer.fit` stopped: ")

    print('Starting the script...')
    # %%
    dataset = [
        FasterSVHN(
            root='../../data/SVHN', split=s,
            transform=transforms.Compose([
                transforms.Resize(32),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.4377, 0.4438, 0.4728],
                    std=[0.1980, 0.2010, 0.1970]
                ),
            ])
        ) for s in ['train', 'test']]

    # dataset = [torch.utils.data.Subset(d, list(range(100))) for d in dataset]
    # for i in range(10):
    #     for d in dataset:
    #         d.dataset.labels[i]=i

    # %%
    w0_accu_grads_list = []
    w1_accu_grads_list = []
    res_per_step = []


    batch_count = np.ceil(len(dataset[0])/__batch_size__/2)

    # %%
    model = CorrResNetPLModel(num_classes=10, resnet_version='resnet18', lr=0.005, )
    # model.load_state_dict(torch.load('data/resnet18_svhn.pth', map_location='cpu'))

    # *****************
    client_epochs_per_round = __epoch_count__
    sim = FLSimulator(
        pl_model=model, num_agents=2, communication_rounds=50, client_epochs_per_round=__epoch_count__,
        batch_size=__batch_size__, dataset_train=dataset[0], dataset_test=dataset[1],
        aggregation_method='fedavg', non_iid_sampling=False, user_logger=None)
    # ****
    sim.run_simulation()



In [ ]:
import pickle
import gzip
from matplotlib import pyplot as plt
import numpy as np
from script_corr import __batch_size__, __epoch_count__, min_window_size

epoch_count = __epoch_count__
batch_count = np.ceil(73200 / __batch_size__ / 2)
min_window_size = min_window_size

file_name = f'corr_res_{0}.pkl.gz'
with gzip.open(file_name, 'rb') as f:
    temp = pickle.load(f)
    corr_y, round_step, nan_ratios = list(zip(*temp))

round_step = np.array(round_step)
# temp_f = lambda idx: (min_window_size+idx)/(batch_count*epoch_count)
#  round_step = temp_f(np.arange(len( corr_y)))
corr_y = np.array(corr_y)


with open('auc.txt') as f:
    auc = eval(f.read())


plt.figure(figsize=(10, 6))

ax1 = plt.gca()
ax2 = ax1.twinx()

ax1.bar(round_step, corr_y, width=0.01)
ax1.xticks(list(range(int(np.floor(round_step.min())), int(np.ceil(round_step.max()) + 1))))
ax1.set_xlabel('Communication round (WINDOWS SIZE = 1.2 Rounds)')
ax1.set_ylabel('Mean Correlation between workers elements')

ax2.plot(np.arange(len(auc)), auc, marker='o', markersize=1, label='val AUC', color='orange')
ax2.set_ylabel('Validation AUC', color='orange')
ax2.tick_params(axis='y', labelcolor='orange')

ax1.tick_params(axis='y', labelcolor='#2E5BBA')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center left')

ax1.grid(True)

plt.title("Correlation between elements of workers' grad in ResNet18 on SVHN dataset")
plt.ylim(0.5, 1)
plt.xlim(1, 25)
plt.grid()
plt.show()


# share of nan values
plt.plot(nan_ratios)
plt.ylim(0, 1)


In [ ]:
# import numpy as np
# import os
# import pickle
# import gzip
# from matplotlib import pyplot as plt
#
# epoch_count = 10
# batch_count = 3
#
# loaded_files = []
# x = []
# for file_name in os.listdir('accum_grads/'):
#     if not file_name.endswith('.pkl.gz'):
#         continue
#     with gzip.open(os.path.join('accum_grads/', file_name), 'rb') as f:
#         res = pickle.load(f)
#         loaded_files.append([None, res])
#     temp = {
#         "round_num": int(file_name.split(',')[0].split('_')[-1]),
#         "epoch_num": int(file_name.split(',')[1].split('_')[1]),
#         "batch_num": int(file_name.split(',')[2].split('_')[1].split('.')[0])
#     }
#     temp = temp['round_num'] + (temp['epoch_num'] + temp['batch_num'] / batch_count) / epoch_count
#     x.append(temp)
#     loaded_files[-1][0] = temp
#
# loaded_files.sort(key=lambda x: x[0])
# y = np.array([x[1] for x in loaded_files])
# x.sort()
# temp = [*np.isnan(y)]
# non_nan_idx = temp[0]
# ratios = [len(non_nan_idx) / len(y)]
# for i in range(1, len(temp)):
#     non_nan_idx = np.logical_or(non_nan_idx, temp[i])
#     ratios += [len(non_nan_idx) / len(y)]
# y = y.T[~non_nan_idx]
#
# y = np.mean(y, axis=0)
# print(y.shape, ratios[0], ratios[-1])
# plt.figure(figsize=(14, 6))
# plt.plot(x, y, )
# plt.xticks([0, 1, 2, 3, 4, ])
# plt.xlabel('Communication round (+epoch/10 + batch/3/10)')
# plt.ylabel('Correlation between workers')
# plt.title("Correlation between workers' gradients in ResNet18 on SVHN dataset")
# plt.grid()
# plt.show()
